# siteHopper v3 — Experiments

This notebook isolates advanced experiments from the core refactor.

It includes the core definitions needed to run independently, then focuses on:
- extended integration tests and SQL diagnostics
- structure extraction + NER
- reference-count and priority-frontier crawl experiments


## Core setup (import + graph-store classes)
These cells are copied from the core notebook so experiments run standalone.


In [ ]:
import os
import re
import time
import math
import json
import sqlite3
import hashlib
import random
import datetime
from dataclasses import dataclass
from typing import Iterable, Optional, List, Dict, Tuple, Any

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from urllib.parse import urlparse, unquote

# Plotly is used for a higher-fidelity interactive graph export
import plotly.graph_objects as go
import plotly.io as pio


In [ ]:
WIKI_BASE_URL = "https://en.wikipedia.org"
WIKI_PATH_PREFIX = "/wiki/"

def utc_now_iso() -> str:
    """UTC timestamp string (seconds precision)"""
    return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()

def canonicalize_page_key(page: str) -> str:
    """Return a canonical Wikipedia page key (title) from many possible inputs.

    Accepts:
      - '/wiki/Title'
      - 'Title' (spaces ok)
      - full URL 'https://en.wikipedia.org/wiki/Title#fragment'
    Returns:
      - 'Title' with spaces replaced by underscores, no fragments/queries.
    """
    if page is None:
        raise ValueError("page cannot be None")
    s = str(page).strip()
    if not s:
        raise ValueError("page cannot be empty")

    # If full URL, extract path
    if s.startswith("http://") or s.startswith("https://"):
        path = urlparse(s).path
    else:
        path = s

    # Strip query/fragment
    path = path.split("#", 1)[0].split("?", 1)[0]

    # Common forms
    if path.startswith(WIKI_PATH_PREFIX):
        key = path[len(WIKI_PATH_PREFIX):]
    else:
        key = path

    key = unquote(key)
    key = key.replace(" ", "_")

    # A last sanity cleanup: remove accidental leading/trailing slashes
    key = key.strip("/")

    return key

def wiki_relpath(page_key: str) -> str:
    return f"{WIKI_PATH_PREFIX}{page_key}"

def wiki_url(page_key: str, base_url: str = WIKI_BASE_URL) -> str:
    return f"{base_url}{WIKI_PATH_PREFIX}{page_key}"


In [ ]:
@dataclass(frozen=True)
class PageRecord:
    page: str
    url: str
    content: Optional[str] = None
    status_code: Optional[int] = None
    fetched_at: Optional[str] = None

@dataclass(frozen=True)
class EdgeRecord:
    src: str
    dst: str
    rel: str = "WIKI_LINK"
    weight: Optional[float] = None
    anchor_text: Optional[str] = None
    discovered_at: Optional[str] = None


class WikiSqliteStore:
    """Efficient SQLite-backed cache + graph store."""

    def __init__(self, db_path: str = "wiki_cache.db", base_url: str = WIKI_BASE_URL):
        self.db_path = db_path
        self.base_url = base_url
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row

        # Performance-oriented pragmas (safe defaults for a local cache)
        self.conn.execute("PRAGMA journal_mode=WAL;")
        self.conn.execute("PRAGMA synchronous=NORMAL;")
        self.conn.execute("PRAGMA foreign_keys=ON;")
        self.conn.execute("PRAGMA temp_store=MEMORY;")

        # In-process cache to reduce repeated SELECTs during crawls
        self._id_cache: Dict[str, int] = {}

        self._init_schema()

    def close(self):
        self.conn.close()

    def _init_schema(self):
        self.conn.executescript("""
        CREATE TABLE IF NOT EXISTS pages (
            page_id       INTEGER PRIMARY KEY,
            page          TEXT NOT NULL UNIQUE,      -- canonical key
            url           TEXT NOT NULL,
            content       TEXT,
            content_sha256 TEXT,
            status_code   INTEGER,
            fetched_at    TEXT
        );

        CREATE TABLE IF NOT EXISTS edges (
            src_page_id   INTEGER NOT NULL,
            dst_page_id   INTEGER NOT NULL,
            rel           TEXT NOT NULL DEFAULT 'WIKI_LINK',
            weight        REAL,
            anchor_text   TEXT,
            discovered_at TEXT,
            PRIMARY KEY (src_page_id, dst_page_id, rel),
            FOREIGN KEY (src_page_id) REFERENCES pages(page_id),
            FOREIGN KEY (dst_page_id) REFERENCES pages(page_id)
        ) WITHOUT ROWID;

        -- For incoming-link queries (optional but cheap)
        CREATE INDEX IF NOT EXISTS idx_edges_dst ON edges(dst_page_id);
        """)
        self.conn.commit()

    # ---------- basic stats ----------
    def db_component_sizes(self) -> Dict[str, int]:
        """Return on-disk sizes of SQLite components (db + wal + shm).

        Useful for empirical storage benchmarks when using WAL mode.
        """
        base = self.db_path
        parts = {
            "db": base,
            "wal": base + "-wal",
            "shm": base + "-shm",
        }
        sizes: Dict[str, int] = {}
        for name, path in parts.items():
            sizes[name] = os.path.getsize(path) if os.path.exists(path) else 0
        return sizes

    def disk_usage_bytes(self) -> int:
        sizes = self.db_component_sizes()
        return int(sum(sizes.values()))


    def counts(self) -> Dict[str, int]:
        cur = self.conn.cursor()
        cur.execute("SELECT COUNT(*) AS n FROM pages")
        pages_n = int(cur.fetchone()["n"])
        cur.execute("SELECT COUNT(*) AS n FROM edges")
        edges_n = int(cur.fetchone()["n"])
        return {"pages": pages_n, "edges": edges_n}

    # ---------- page id helpers ----------
    def _select_page_ids(self, keys: List[str]) -> Dict[str, int]:
        """Fetch page_id for each key (chunked to avoid SQLite parameter limits)."""
        out: Dict[str, int] = {}
        keys = [k for k in keys if k]
        if not keys:
            return out

        # SQLite default variable limit is often 999; keep headroom.
        CHUNK = 900
        cur = self.conn.cursor()
        for i in range(0, len(keys), CHUNK):
            chunk = keys[i:i+CHUNK]
            qs = ",".join(["?"] * len(chunk))
            cur.execute(f"SELECT page, page_id FROM pages WHERE page IN ({qs})", chunk)
            for row in cur.fetchall():
                out[row["page"]] = int(row["page_id"])
        return out

    def _ensure_pages_exist(self, keys: Iterable[str]):
        """Insert placeholder page rows for any unknown keys."""
        keys = [canonicalize_page_key(k) for k in keys]
        keys = list(dict.fromkeys(keys))  # stable dedup

        # fast path: if many already cached
        missing = [k for k in keys if k not in self._id_cache]
        if not missing:
            return

        # Insert placeholders (url only). Content comes later via upsert.
        cur = self.conn.cursor()
        cur.executemany(
            "INSERT OR IGNORE INTO pages(page, url) VALUES(?, ?)",
            [(k, wiki_url(k, self.base_url)) for k in missing]
        )

        # Load ids into cache
        ids = self._select_page_ids(missing)
        self._id_cache.update(ids)

    def get_page(self, key: str) -> Optional[sqlite3.Row]:
        key = canonicalize_page_key(key)
        cur = self.conn.cursor()
        cur.execute(
            """SELECT page_id, page, url, content, content_sha256, status_code, fetched_at
               FROM pages WHERE page=?""",
            (key,)
        )
        return cur.fetchone()

    def neighbors(self, key: str, limit: Optional[int] = None) -> List[str]:
        key = canonicalize_page_key(key)

        # Resolve src id (cache → db)
        src_id = self._id_cache.get(key)
        if src_id is None:
            row = self.get_page(key)
            if row is None:
                return []
            src_id = int(row["page_id"])
            self._id_cache[key] = src_id

        cur = self.conn.cursor()
        sql = """SELECT p.page
                 FROM edges e
                 JOIN pages p ON p.page_id = e.dst_page_id
                 WHERE e.src_page_id = ?
                 ORDER BY e.dst_page_id"""
        params: Tuple[Any, ...]
        if limit is not None:
            sql += " LIMIT ?"
            params = (src_id, int(limit))
        else:
            params = (src_id,)
        cur.execute(sql, params)
        return [r["page"] for r in cur.fetchall()]

    # ---------- writes ----------
    def bulk_write(self, pages: List[PageRecord], edges: List[EdgeRecord]):
        """Write pages and edges in a single transaction (fast path for crawls)."""
        # Canonicalize and normalize inputs
        norm_pages: List[PageRecord] = []
        for p in pages:
            key = canonicalize_page_key(p.page)
            url = p.url or wiki_url(key, self.base_url)
            fetched_at = p.fetched_at or (utc_now_iso() if p.content is not None else None)
            norm_pages.append(PageRecord(
                page=key,
                url=url,
                content=p.content,
                status_code=p.status_code,
                fetched_at=fetched_at
            ))

        norm_edges: List[EdgeRecord] = []
        for e in edges:
            src = canonicalize_page_key(e.src)
            dst = canonicalize_page_key(e.dst)
            norm_edges.append(EdgeRecord(
                src=src,
                dst=dst,
                rel=e.rel or "WIKI_LINK",
                weight=e.weight,
                anchor_text=e.anchor_text,
                discovered_at=e.discovered_at or utc_now_iso()
            ))

        # Collect all keys we must ensure exist
        all_keys: List[str] = []
        all_keys.extend([p.page for p in norm_pages])
        for e in norm_edges:
            all_keys.append(e.src)
            all_keys.append(e.dst)
        all_keys = list(dict.fromkeys(all_keys))

        cur = self.conn.cursor()
        cur.execute("BEGIN")
        try:
            # 1) Ensure placeholders for everything (url only)
            cur.executemany(
                "INSERT OR IGNORE INTO pages(page, url) VALUES(?, ?)",
                [(k, wiki_url(k, self.base_url)) for k in all_keys]
            )

            # 2) Upsert content-bearing pages
            upsert_rows = []
            for p in norm_pages:
                if p.content is None:
                    continue
                h = sha256_text(p.content)
                upsert_rows.append((p.page, p.url, p.content, h, p.status_code, p.fetched_at))

            cur.executemany(
                """INSERT INTO pages(page, url, content, content_sha256, status_code, fetched_at)
                   VALUES(?,?,?,?,?,?)
                   ON CONFLICT(page) DO UPDATE SET
                       url=excluded.url,
                       content=excluded.content,
                       content_sha256=excluded.content_sha256,
                       status_code=excluded.status_code,
                       fetched_at=excluded.fetched_at
                """,
                upsert_rows
            )

            # 3) Load ids for keys touched in this batch
            ids = self._select_page_ids(all_keys)
            self._id_cache.update(ids)

            # 4) Insert edges by ids
            edge_rows = []
            for e in norm_edges:
                src_id = ids.get(e.src)
                dst_id = ids.get(e.dst)
                if src_id is None or dst_id is None:
                    continue
                edge_rows.append((src_id, dst_id, e.rel, e.weight, e.anchor_text, e.discovered_at))

            # Dedup edges within the batch to reduce insert work
            if edge_rows:
                edge_rows = list(dict.fromkeys(edge_rows))

            cur.executemany(
                """INSERT OR IGNORE INTO edges(
                       src_page_id, dst_page_id, rel, weight, anchor_text, discovered_at
                   ) VALUES(?,?,?,?,?,?)""",
                edge_rows
            )

            self.conn.commit()
        except Exception:
            self.conn.rollback()
            raise

    def upsert_page(self, page: PageRecord):
        self.bulk_write([page], [])

    def insert_edges(self, edges: List[EdgeRecord]):
        self.bulk_write([], edges)


In [ ]:
class CrawlBatchWriter:
    def __init__(self, store: WikiSqliteStore, batch_pages: int = 50, batch_edges: int = 10_000):
        self.store = store
        self.batch_pages = int(batch_pages)
        self.batch_edges = int(batch_edges)
        self._pages: List[PageRecord] = []
        self._edges: List[EdgeRecord] = []

    def add_page(self, page: PageRecord):
        self._pages.append(page)
        self._maybe_flush()

    def add_edges(self, edges: Iterable[EdgeRecord]):
        self._edges.extend(list(edges))
        self._maybe_flush()

    def _maybe_flush(self):
        if len(self._pages) >= self.batch_pages or len(self._edges) >= self.batch_edges:
            self.flush()

    def flush(self):
        if not self._pages and not self._edges:
            return
        self.store.bulk_write(self._pages, self._edges)
        self._pages.clear()
        self._edges.clear()

    def close(self):
        self.flush()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()


In [ ]:
import requests
from bs4 import BeautifulSoup

class WikipediaScraper:
    """Network scraper: fetches Wikipedia page text + outlinks."""
    def __init__(self, base_url: str = WIKI_BASE_URL, timeout: int = 15):
        self.base_url = base_url
        self.timeout = timeout
        self.headers = {
            "User-Agent": "Mozilla/5.0 (siteHopper research crawler; +https://example.org)"
        }

    def _clean_wiki_hrefs(self, hrefs: Iterable[str]) -> List[str]:
        out = []
        for href in hrefs:
            if not href:
                continue
            if not href.startswith("/wiki/"):
                continue
            href = href.split("#", 1)[0].split("?", 1)[0]
            # Drop non-article namespaces (File:, Category:, Help:, etc.)
            if ":" in href[len("/wiki/"):]:
                continue
            key = canonicalize_page_key(href)
            out.append(key)
        # stable dedup
        return list(dict.fromkeys(out))

    def fetch_page(self, page: str) -> Tuple[str, List[str], int]:
        key = canonicalize_page_key(page)
        url = wiki_url(key, self.base_url)
        resp = requests.get(url, headers=self.headers, timeout=self.timeout)
        status = int(resp.status_code)

        if status != 200:
            return "", [], status

        soup = BeautifulSoup(resp.content, "html.parser")

        # Extract outlinks
        hrefs = [a.get("href") for a in soup.find_all("a", href=True)]
        outlinks = self._clean_wiki_hrefs(hrefs)

        # Strip noisy elements for content extraction
        for element in soup(["table", "ul", "ol", "dl", "sup", "span"]):
            element.decompose()

        body = soup.find("div", {"id": "bodyContent"})
        if body:
            paragraphs = body.find_all("p")
            content = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
        else:
            content = ""

        return content, outlinks, status


class FakeWikipediaScraper:
    """Deterministic offline generator for content + outlinks.

    Given a key, it produces:
      - content: pseudo-random text
      - outlinks: deterministic list of other page keys
    """
    def __init__(self, n_pages: int = 5000, out_degree: int = 12, seed: int = 123):
        self.n_pages = int(n_pages)
        self.out_degree = int(out_degree)
        self.seed = int(seed)
        self.vocab = [f"w{i}" for i in range(1000)]

    def _rng_for_key(self, key: str) -> random.Random:
        h = int(hashlib.sha256((key + str(self.seed)).encode("utf-8")).hexdigest(), 16)
        return random.Random(h)

    def fetch_page(self, page: str) -> Tuple[str, List[str], int]:
        key = canonicalize_page_key(page)
        rng = self._rng_for_key(key)

        # content length ~ 800-1200 words
        words = rng.choices(self.vocab, k=rng.randint(800, 1200))
        content = " ".join(words)

        # deterministic outlinks
        outlinks = []
        for _ in range(self.out_degree):
            j = rng.randrange(self.n_pages)
            outlinks.append(f"Page_{j}")
        outlinks = list(dict.fromkeys(outlinks))  # dedup

        return content, outlinks, 200


In [ ]:
class WikipediaPage:
    def __init__(self, page: str, store: WikiSqliteStore, scraper, batch: Optional[CrawlBatchWriter] = None):
        self.store = store
        self.scraper = scraper
        self.key = canonicalize_page_key(page)
        self.url = wiki_url(self.key, self.store.base_url)
        self._content: Optional[str] = None
        self._outlinks: Optional[List[str]] = None  # set when freshly fetched (so crawling doesn't depend on DB flush)

        row = self.store.get_page(self.key)
        if row is not None and row["content"]:
            self._content = row["content"]
            self._outlinks = None  # read outlinks from DB on demand
        else:
            content, outlinks, status = self.scraper.fetch_page(self.key)
            self._content = content
            self._outlinks = outlinks

            now = utc_now_iso()
            page_rec = PageRecord(
                page=self.key,
                url=self.url,
                content=content,
                status_code=status,
                fetched_at=now,
            )
            edge_recs = [EdgeRecord(src=self.key, dst=dst, discovered_at=now) for dst in outlinks]

            if batch is not None:
                batch.add_page(page_rec)
                batch.add_edges(edge_recs)
            else:
                self.store.bulk_write([page_rec], edge_recs)

    def get_page_name(self) -> str:
        return self.key

    def get_content(self) -> str:
        return self._content or ""

    def get_hyperlinks(self, limit: Optional[int] = None) -> List[str]:
        # If newly fetched, return in-memory outlinks immediately (batch may not be flushed yet).
        if self._outlinks is not None:
            if limit is None:
                return list(self._outlinks)
            return list(self._outlinks)[: int(limit)]
        return self.store.neighbors(self.key, limit=limit)


In [ ]:
class WikiNetworkBuilder:
    def __init__(self, start_page: str, store: WikiSqliteStore, scraper):
        self.store = store
        self.scraper = scraper
        self.start_key = canonicalize_page_key(start_page)

    def crawl(self, depth: int = 2, max_connections: int = 10, seed: int = 0,
              batch_pages: int = 50, batch_edges: int = 10_000) -> Dict[str, Any]:
        rng = random.Random(seed)
        visited = set()
        frontier = [self.start_key]
        n_fetched = 0

        with CrawlBatchWriter(self.store, batch_pages=batch_pages, batch_edges=batch_edges) as batch:
            for d in range(depth):
                next_frontier = []
                for key in frontier:
                    if key in visited:
                        continue
                    visited.add(key)

                    # Ensure cached (fetch if needed)
                    page = WikipediaPage(key, store=self.store, scraper=self.scraper, batch=batch)
                    n_fetched += 1

                    outlinks = page.get_hyperlinks()
                    if max_connections is not None and len(outlinks) > max_connections:
                        outlinks = rng.sample(outlinks, k=max_connections)

                    next_frontier.extend(outlinks)

                frontier = next_frontier

        return {
            "start": self.start_key,
            "depth": depth,
            "visited": len(visited),
            "fetched": n_fetched,
        }

    def extract_subgraph(self, center: str, max_nodes: int = 200) -> nx.DiGraph:
        """Extract a small subgraph around `center` for visualization."""
        center = canonicalize_page_key(center)
        g = nx.DiGraph()
        q = [center]
        seen = set()

        while q and len(seen) < max_nodes:
            u = q.pop(0)
            if u in seen:
                continue
            seen.add(u)
            g.add_node(u)
            nbrs = self.store.neighbors(u, limit=50)
            for v in nbrs:
                g.add_edge(u, v)
                if v not in seen and len(seen) + len(q) < max_nodes:
                    q.append(v)
        return g


## 11) Extended integration tests + SQL diagnostics + structure + NER

This section demonstrates:
- end-to-end page grabbing + cache behavior
- SQL introspection queries for graph/page storage
- structured extraction from live Wikipedia HTML
- entity extraction from page content (transformers with regex fallback)


In [ ]:
import time
import tempfile
from pathlib import Path

import pandas as pd

def _sql_df(conn, sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

def run_extended_refactor_tests(
    mode='fake',
    start_page='Page_0',
    depth=3,
    max_connections=6,
    seed=7,
    n_pages=3000,
    out_degree=12,
):
    """
    mode='fake': deterministic, fast, offline-capable tests
    mode='live': real Wikipedia page grabbing tests
    """
    if mode not in {'fake', 'live'}:
        raise ValueError("mode must be 'fake' or 'live'")

    with tempfile.TemporaryDirectory() as td:
        db = str(Path(td) / 'extended_refactor.db')
        store = WikiSqliteStore(db_path=db)

        if mode == 'fake':
            scraper = FakeWikipediaScraper(n_pages=n_pages, out_degree=out_degree, seed=seed)
            start_key = canonicalize_page_key(start_page)
        else:
            scraper = WikipediaScraper()
            if start_page == 'Page_0':
                start_page = 'Philosophy'
            start_key = canonicalize_page_key(start_page)

        # --- page grabbing smoke test ---
        t0 = time.time()
        content, outlinks, status = scraper.fetch_page(start_key)
        fetch_ms = round((time.time() - t0) * 1000, 2)

        fetch_ok = (
            isinstance(content, str)
            and isinstance(outlinks, list)
            and isinstance(status, int)
        )

        # --- cache behavior check ---
        p1 = WikipediaPage(start_key, store=store, scraper=scraper)
        counts_after_first = store.counts().copy()

        p2 = WikipediaPage(start_key, store=store, scraper=scraper)
        counts_after_second = store.counts().copy()

        cached_second_read_no_growth = (
            counts_after_second['pages'] == counts_after_first['pages']
            and counts_after_second['edges'] == counts_after_first['edges']
        )

        # --- crawl builder test ---
        builder = WikiNetworkBuilder(start_page=start_key, store=store, scraper=scraper)
        crawl_stats = builder.crawl(
            depth=depth,
            max_connections=max_connections,
            seed=seed,
            batch_pages=50,
            batch_edges=20_000,
        )

        counts_after_crawl = store.counts()
        subg = builder.extract_subgraph(center=start_key, max_nodes=120)

        # --- SQL diagnostics ---
        q_counts = _sql_df(store.conn, "SELECT COUNT(*) AS pages FROM pages")
        q_edges = _sql_df(store.conn, "SELECT COUNT(*) AS edges FROM edges")

        q_outdegree = _sql_df(
            store.conn,
            """
            SELECT p.page, COUNT(e.dst_page_id) AS out_degree
            FROM pages p
            LEFT JOIN edges e ON p.page_id = e.src_page_id
            GROUP BY p.page_id
            ORDER BY out_degree DESC, p.page
            LIMIT 15
            """
        )

        q_indegree = _sql_df(
            store.conn,
            """
            SELECT p.page, COUNT(e.src_page_id) AS in_degree
            FROM pages p
            LEFT JOIN edges e ON p.page_id = e.dst_page_id
            GROUP BY p.page_id
            ORDER BY in_degree DESC, p.page
            LIMIT 15
            """
        )

        q_recent = _sql_df(
            store.conn,
            """
            SELECT page, status_code, fetched_at, LENGTH(content) AS content_len
            FROM pages
            ORDER BY fetched_at DESC
            LIMIT 15
            """
        )

        summary = {
            'mode': mode,
            'start_key': start_key,
            'fetch_ok': fetch_ok,
            'fetch_status': status,
            'fetch_ms': fetch_ms,
            'initial_outlinks': len(outlinks),
            'cached_second_read_no_growth': cached_second_read_no_growth,
            'counts_after_first': counts_after_first,
            'counts_after_crawl': counts_after_crawl,
            'crawl_stats': crawl_stats,
            'subgraph_nodes': subg.number_of_nodes(),
            'subgraph_edges': subg.number_of_edges(),
            'disk_usage_bytes': store.disk_usage_bytes(),
        }

        store.close()

        return {
            'summary': summary,
            'q_counts': q_counts,
            'q_edges': q_edges,
            'q_outdegree': q_outdegree,
            'q_indegree': q_indegree,
            'q_recent': q_recent,
        }

# Fast deterministic run
ext_results = run_extended_refactor_tests(mode='fake', start_page='Philosophy', depth=1, max_connections=6, seed=7)
print('Summary:', ext_results['summary'])
display(ext_results['q_counts'])
display(ext_results['q_edges'])
display(ext_results['q_outdegree'])
display(ext_results['q_indegree'])
display(ext_results['q_recent'])

# Optional live test:
live_results = run_extended_refactor_tests(mode='live', start_page='Artificial_intelligence', depth=2, max_connections=4, seed=7)
print('Live summary:', live_results['summary'])


In [ ]:
import requests
from bs4 import BeautifulSoup

def extract_wikipedia_structure(page: str) -> dict:
    key = canonicalize_page_key(page)
    url = wiki_url(key)

    headers = {'User-Agent': 'Mozilla/5.0 (siteHopper structure extractor)'}
    resp = requests.get(url, headers=headers, timeout=20)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.content, 'html.parser')

    title_tag = soup.select_one('#firstHeading')
    title = title_tag.get_text(' ', strip=True) if title_tag else key

    body = soup.select_one('#bodyContent')

    lead_paragraphs = []
    if body:
        for p in body.find_all('p')[:4]:
            text = p.get_text(' ', strip=True)
            if text:
                lead_paragraphs.append(text)

    section_headings = []
    if body:
        for h in body.select('h2 .mw-headline, h3 .mw-headline'):
            sec = h.get_text(' ', strip=True)
            if sec:
                section_headings.append(sec)

    infobox = {}
    ibox = soup.select_one('table.infobox')
    if ibox:
        for row in ibox.select('tr'):
            th = row.find('th')
            td = row.find('td')
            if th and td:
                k = th.get_text(' ', strip=True)
                v = td.get_text(' ', strip=True)
                if k and v:
                    infobox[k] = v

    categories = [
        a.get_text(' ', strip=True)
        for a in soup.select('#mw-normal-catlinks ul li a')
        if a.get_text(' ', strip=True)
    ]

    return {
        'key': key,
        'url': url,
        'title': title,
        'lead_paragraphs': lead_paragraphs,
        'section_headings': section_headings,
        'infobox': infobox,
        'categories': categories,
    }

# Example structure extraction
structure = extract_wikipedia_structure('Philosophy')
print('Title:', structure['title'])
print('URL:', structure['url'])
print('Lead paragraphs:', len(structure['lead_paragraphs']))
print('Sections:', len(structure['section_headings']))
print('Infobox fields:', len(structure['infobox']))
print('Categories:', len(structure['categories']))
print('First 10 sections:', structure['section_headings'][:10])


In [ ]:
import re
import tempfile
from pathlib import Path

import pandas as pd

_NER_PIPELINE = None

def _regex_entity_fallback(text: str, top_n: int = 30) -> pd.DataFrame:
    # Heuristic fallback: multi-word capitalized phrases
    pattern = r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b'
    matches = re.findall(pattern, text or '')
    if not matches:
        return pd.DataFrame(columns=['entity', 'count'])
    vc = pd.Series(matches).value_counts().head(top_n)
    return vc.rename_axis('entity').reset_index(name='count')

def _get_transformer_ner(model: str = 'dslim/bert-base-NER'):
    global _NER_PIPELINE
    if _NER_PIPELINE is None:
        from transformers import pipeline
        _NER_PIPELINE = pipeline('ner', model=model, aggregation_strategy='simple')
    return _NER_PIPELINE

def run_ner_for_page(
    page='Evolution',
    max_chars=4000,
    top_n=30,
    prefer_transformer=True,
    mode='live',
):
    """
    Loads/caches a page, then runs NER on content.
    mode='fake' uses FakeWikipediaScraper (offline deterministic).
    mode='live' uses real WikipediaScraper.
    """
    with tempfile.TemporaryDirectory() as td:
        db = str(Path(td) / 'ner.db')
        store = WikiSqliteStore(db_path=db)

        if mode == 'fake':
            scraper = FakeWikipediaScraper(n_pages=5000, out_degree=12, seed=13)
            key = canonicalize_page_key(page if str(page).startswith('Page_') else 'Page_0')
        else:
            scraper = WikipediaScraper()
            key = canonicalize_page_key(page)

        wp = WikipediaPage(key, store=store, scraper=scraper)
        text = (wp.get_content() or '')[:max_chars]

        if not text.strip():
            store.close()
            return {'mode': 'none', 'key': key, 'text_chars': 0, 'entities_df': pd.DataFrame()}

        if prefer_transformer:
            try:
                ner = _get_transformer_ner()
                ents = ner(text)
                df = pd.DataFrame(ents)

                if not df.empty and 'word' in df.columns:
                    grp_col = 'entity_group' if 'entity_group' in df.columns else ('entity' if 'entity' in df.columns else None)
                    if grp_col and 'score' in df.columns:
                        df = (
                            df.groupby(['word', grp_col], as_index=False)
                              .agg(avg_score=('score', 'mean'), mentions=('word', 'count'))
                              .sort_values(['mentions', 'avg_score'], ascending=False)
                              .head(top_n)
                        )
                out_mode = 'transformers'
            except Exception as e:
                print('Transformer NER unavailable, using regex fallback:', e)
                df = _regex_entity_fallback(text, top_n=top_n)
                out_mode = 'regex_fallback'
        else:
            df = _regex_entity_fallback(text, top_n=top_n)
            out_mode = 'regex_fallback'

        store.close()
        return {'mode': out_mode, 'key': key, 'text_chars': len(text), 'entities_df': df}

# Example NER run (live). For fully offline demo, use mode='fake'.
ner_out = run_ner_for_page(page='Scotland', max_chars=3500, top_n=25, prefer_transformer=True, mode='live')
print('NER mode:', ner_out['mode'])
print('Page key:', ner_out['key'])
print('Text chars:', ner_out['text_chars'])
display(ner_out['entities_df'])
ner_out = run_ner_for_page(page='Aristotle', max_chars=3500, top_n=50, prefer_transformer=True, mode='live')
print('NER mode:', ner_out['mode'])
print('Page key:', ner_out['key'])
print('Text chars:', ner_out['text_chars'])
display(ner_out['entities_df'])


In [ ]:
ner_out = run_ner_for_page(page='Scotland', max_chars=3500, top_n=25, prefer_transformer=True, mode='live')
print('NER mode:', ner_out['mode'])
print('Page key:', ner_out['key'])
print('Text chars:', ner_out['text_chars'])
display(ner_out['entities_df'])

In [ ]:
from collections import defaultdict, Counter
import random
import pandas as pd

def crawl_with_reference_counts(builder, depth=3, max_connections=10, seed=0, expand_unique_nodes=True):
    """
    Counts how often links are referenced at each crawl depth.

    Returns:
      edge_counts_df: depth, src, dst, refs
      dst_counts_df:  depth, dst, refs
      total_dst_df:   dst, total_refs (across all depths)
    """
    rng = random.Random(seed)
    frontier = [builder.start_key]
    visited = set()

    # depth -> Counter((src,dst) -> count)
    edge_counts_by_depth = defaultdict(Counter)
    # depth -> Counter(dst -> count)
    dst_counts_by_depth = defaultdict(Counter)

    for d in range(depth):
        next_frontier = []
        for src in frontier:
            if expand_unique_nodes and src in visited:
                continue
            visited.add(src)

            page = WikipediaPage(src, store=builder.store, scraper=builder.scraper)
            outlinks = page.get_hyperlinks()

            if max_connections is not None and len(outlinks) > max_connections:
                outlinks = rng.sample(outlinks, k=max_connections)

            # Count every reference observed at this layer
            for dst in outlinks:
                layer = d + 1
                edge_counts_by_depth[layer][(src, dst)] += 1
                dst_counts_by_depth[layer][dst] += 1

            next_frontier.extend(outlinks)

        frontier = next_frontier

    edge_rows = []
    for layer, ctr in edge_counts_by_depth.items():
        for (src, dst), refs in ctr.items():
            edge_rows.append({"depth": layer, "src": src, "dst": dst, "refs": refs})

    dst_rows = []
    for layer, ctr in dst_counts_by_depth.items():
        for dst, refs in ctr.items():
            dst_rows.append({"depth": layer, "dst": dst, "refs": refs})

    edge_counts_df = pd.DataFrame(edge_rows).sort_values(["depth", "refs"], ascending=[True, False])
    dst_counts_df = pd.DataFrame(dst_rows).sort_values(["depth", "refs"], ascending=[True, False])

    if not dst_counts_df.empty:
        total_dst_df = (
            dst_counts_df.groupby("dst", as_index=False)["refs"]
            .sum()
            .rename(columns={"refs": "total_refs"})
            .sort_values("total_refs", ascending=False)
        )
    else:
        total_dst_df = pd.DataFrame(columns=["dst", "total_refs"])

    return edge_counts_df, dst_counts_df, total_dst_df


In [ ]:
# Example setup
store = WikiSqliteStore("ref_counts.db")
scraper = WikipediaScraper()  # or FakeWikipediaScraper(...)
builder = WikiNetworkBuilder(start_page="Artificial_intelligence", store=store, scraper=scraper)

edge_df, dst_df, total_df = crawl_with_reference_counts(
    builder,
    depth=4,
    max_connections=8,
    seed=7,
    expand_unique_nodes=True
)

display(dst_df.head(20))    # per-depth counts
display(total_df.head(20))  # overall most-referenced links


In [ ]:
store.conn.execute("""
CREATE TABLE IF NOT EXISTS crawl_ref_counts (
    run_id TEXT,
    depth INTEGER,
    src TEXT,
    dst TEXT,
    refs INTEGER,
    PRIMARY KEY (run_id, depth, src, dst)
)
""")

run_id = "run_001"
store.conn.executemany(
    "INSERT OR REPLACE INTO crawl_ref_counts(run_id, depth, src, dst, refs) VALUES (?, ?, ?, ?, ?)",
    [(run_id, int(r.depth), r.src, r.dst, int(r.refs)) for r in edge_df.itertuples(index=False)]
)
store.conn.commit()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1) Build crawl data first (if you have not already)
store = WikiSqliteStore("analysis.db")
scraper = WikipediaScraper()  # or FakeWikipediaScraper(...)
builder = WikiNetworkBuilder(start_page="Aristotle", store=store, scraper=scraper)
crawl_stats = builder.crawl(depth=2, max_connections=10, seed=7)
print("crawl_stats:", crawl_stats)
print("store_counts:", store.counts())

# 2) SQL: "most visited" proxy = highest in-degree (most referenced by other pages)
q_top_in_degree = """
SELECT p.page,
       COUNT(e.src_page_id) AS in_degree
FROM pages p
LEFT JOIN edges e
  ON p.page_id = e.dst_page_id
GROUP BY p.page_id
ORDER BY in_degree DESC, p.page
LIMIT 20;
"""
top_in = pd.read_sql_query(q_top_in_degree, store.conn)
display(top_in)

# 3) SQL: top outgoing hubs (optional)
q_top_out_degree = """
SELECT p.page,
       COUNT(e.dst_page_id) AS out_degree
FROM pages p
LEFT JOIN edges e
  ON p.page_id = e.src_page_id
GROUP BY p.page_id
ORDER BY out_degree DESC, p.page
LIMIT 20;
"""
top_out = pd.read_sql_query(q_top_out_degree, store.conn)
display(top_out)

# 4) Visualize query results
plt.figure(figsize=(12, 6))
sns.barplot(data=top_in.head(15), x="in_degree", y="page", palette="Blues_r")
plt.title("Most Referenced Pages in Crawled Network (In-Degree)")
plt.xlabel("In-Degree (times linked-to)")
plt.ylabel("Page")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 6))
sns.barplot(data=top_out.head(15), x="out_degree", y="page", palette="Greens_r")
plt.title("Top Outgoing-Link Hubs in Crawled Network (Out-Degree)")
plt.xlabel("Out-Degree (links out)")
plt.ylabel("Page")
plt.tight_layout()
plt.show()


In [ ]:
import heapq
import math
import random
import itertools
from collections import Counter
import pandas as pd

class FrontierPriorityQueue:
    """
    Priority frontier that:
    - stockpiles candidates from many parents
    - counts repeated references
    - filters already-stored pages when popping
    """
    def __init__(self, store, max_depth, w_ref=2.0, w_novelty=3.0, w_depth=1.0):
        self.store = store
        self.max_depth = int(max_depth)
        self.w_ref = float(w_ref)
        self.w_novelty = float(w_novelty)
        self.w_depth = float(w_depth)

        self._heap = []
        self._tick = itertools.count()
        self.ref_counts = Counter()
        self._stored_cache = {}

    def __len__(self):
        return len(self._heap)

    def _is_stored(self, key: str) -> bool:
        key = canonicalize_page_key(key)
        if key in self._stored_cache:
            return self._stored_cache[key]
        row = self.store.get_page(key)
        ok = bool(row is not None and row["content"] is not None and len(row["content"]) > 0)
        self._stored_cache[key] = ok
        return ok

    def _score(self, key: str, depth: int) -> float:
        refs = self.ref_counts[key]
        novelty = 0.0 if self._is_stored(key) else 1.0
        depth_term = 1.0 / (1.0 + depth)
        return (
            self.w_ref * math.log1p(refs) +
            self.w_novelty * novelty +
            self.w_depth * depth_term
        )

    def add_candidate(self, key: str, parent: str = None, depth: int = 0):
        key = canonicalize_page_key(key)
        if depth > self.max_depth:
            return
        self.ref_counts[key] += 1
        score = self._score(key, depth)
        heapq.heappush(self._heap, (-score, next(self._tick), key, parent, depth))

    def pop_next(self, visited: set):
        while self._heap:
            neg_score, _, key, parent, depth = heapq.heappop(self._heap)

            # hard filters
            if key in visited:
                continue
            if self._is_stored(key):
                continue

            return {
                "key": key,
                "parent": parent,
                "depth": depth,
                "score": -neg_score,
                "ref_count": self.ref_counts[key],
            }
        return None


def crawl_with_priority_frontier(
    start_page: str,
    store: WikiSqliteStore,
    scraper,
    depth: int = 3,
    max_connections: int = 8,
    seed: int = 7,
    batch_pages: int = 50,
    batch_edges: int = 20000,
):
    rng = random.Random(seed)
    start_key = canonicalize_page_key(start_page)

    frontier = FrontierPriorityQueue(store=store, max_depth=depth)
    visited = set()
    crawl_log = []

    frontier.add_candidate(start_key, parent=None, depth=0)

    with CrawlBatchWriter(store, batch_pages=batch_pages, batch_edges=batch_edges) as batch:
        while len(frontier) > 0:
            item = frontier.pop_next(visited)
            if item is None:
                break

            key = item["key"]
            d = item["depth"]

            page = WikipediaPage(key, store=store, scraper=scraper, batch=batch)
            visited.add(key)

            outlinks = page.get_hyperlinks()
            if max_connections is not None and len(outlinks) > max_connections:
                outlinks = rng.sample(outlinks, k=max_connections)

            crawl_log.append({
                "page": key,
                "depth": d,
                "score": item["score"],
                "ref_count": item["ref_count"],
                "outlinks_used": len(outlinks),
            })

            if d >= depth:
                continue

            for dst in outlinks:
                frontier.add_candidate(dst, parent=key, depth=d + 1)

    crawl_df = pd.DataFrame(crawl_log)
    ref_df = (
        pd.DataFrame(
            [{"page": k, "references_seen": v} for k, v in frontier.ref_counts.items()]
        )
        .sort_values("references_seen", ascending=False)
        .reset_index(drop=True)
    )

    return crawl_df, ref_df


In [ ]:
store = WikiSqliteStore("priority_frontier.db")
scraper = WikipediaScraper()  # or FakeWikipediaScraper(...)
crawl_df, ref_df = crawl_with_priority_frontier(
    start_page="Artificial_intelligence",
    store=store,
    scraper=scraper,
    depth=3,
    max_connections=3,
    seed=7,
)

display(crawl_df.head(20))
display(ref_df.head(20))  # most-referenced candidates across layers
print(store.counts())
store.close()


score(v)=αlog(1+refs(v))+β⋅1[not_stored(v)]+
γ * (1/(1+depth(v)))

references_seen rewards consensus (many parents point to same page).
novelty rewards uncached pages (new information gain).
depth penalty avoids deep, noisy expansion too early.
Tune weights by objective:
discovery-heavy: increase w_novelty
authority/hub-heavy: increase w_ref
local-context-heavy: increase w_depth (shallower first)
​
